In [ ]:
import pandas as pd
import numpy as np
from faker import Faker

print("Environment ready")

In [ ]:
borrowers = pd.read_csv("../data/borrowers.csv")
loans = pd.read_csv("../data/loans.csv")
repayments = pd.read_csv("../data/repayments.csv")

borrowers.head()

In [ ]:
loan_features = repayments.groupby("loan_id").agg(
    avg_days_past_due=("days_past_due","mean"),
    max_days_past_due=("days_past_due","max"),
    late_payment_count=("days_past_due",lambda x:(x>0).sum()),
    missed_payment_count=("days_past_due",lambda x:(x>=30).sum()),
    total_payments=("emi_number","count")
).reset_index()

loan_features.head()

In [ ]:
loan_features["late_payment_ratio"] = (
    loan_features["late_payment_count"] /
    loan_features["total_payments"]
)

loan_features.head()

In [ ]:
loan_features = loan_features.merge(loans, on="loan_id")

loan_features.head()

In [ ]:
loan_features = loan_features.merge(
    borrowers,
    on="borrower_id",
    how="left"
)

In [ ]:
loan_features["emi_income_ratio"] = (
    loan_features["emi_amount"] /
    loan_features["income"]
)

In [ ]:
loan_features.to_csv("../data/loan_features.csv", index=False)

In [ ]:
loan_features["default_flag"] = (
    loan_features["max_days_past_due"] >= 90
).astype(int)

loan_features[["loan_id","max_days_past_due","default_flag"]].head()

In [ ]:
loan_features["default_flag"].value_counts()

In [ ]:
loan_features["default_flag"].mean()

In [ ]:
loan_features.to_csv("../data/loan_features_labeled.csv", index=False)

In [ ]:
loan_features["default_flag"].value_counts()

In [ ]:
features = [
    "late_payment_ratio",
    "missed_payment_count",
    "emi_income_ratio",
    "income",
    "credit_score",
    "loan_amount",
    "tenure"
]

X = loan_features[features]

y = loan_features["default_flag"]

X.head()

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train,y_train)

In [ ]:
predictions = model.predict(X_test)

predictions[:10]

In [ ]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test,predictions)

In [ ]:
from sklearn.metrics import roc_auc_score

probabilities = model.predict_proba(X_test)[:,1]

roc_auc_score(y_test,probabilities)

In [ ]:
loan_features["default_probability"] = model.predict_proba(X)[:,1]

loan_features[[
    "loan_id",
    "default_probability"
]].head()

In [ ]:
loan_features["risk_score"] = (
    (1 - loan_features["default_probability"]) * 600 + 300
).astype(int)

In [ ]:
loan_features.head()

In [ ]:
def risk_tier(score):

    if score >= 750:
        return "A - Low Risk"

    elif score >= 650:
        return "B - Medium Risk"

    elif score >= 550:
        return "C - High Risk"

    else:
        return "D - Very High Risk"


loan_features["risk_tier"] = loan_features["risk_score"].apply(risk_tier)


In [ ]:
loan_features["risk_tier"].value_counts()

In [ ]:
def decision(tier):

    if tier == "A - Low Risk":
        return "Approve"

    elif tier == "B - Medium Risk":
        return "Approve with conditions"

    elif tier == "C - High Risk":
        return "Manual review"

    else:
        return "Reject"


loan_features["decision"] = loan_features["risk_tier"].apply(decision)

In [ ]:
loan_features[[
    "risk_score",
    "risk_tier",
    "decision"
]].head()

In [ ]:
!py -m pip install openai langchain

In [ ]:
correlations = loan_features[["late_payment_ratio",
        "missed_payment_count",
        "emi_income_ratio",
        "income",
        "credit_score",
        "default_flag"]].corr()

correlations["default_flag"].sort_values(ascending=False)

In [ ]:
risk_drivers = correlations["default_flag"].drop("default_flag")

top_positive = risk_drivers.sort_values(ascending=False).head(3)

top_negative = risk_drivers.sort_values().head(3)

print("Top Risk Drivers:")
print(top_positive)

print("\nRisk Reducing Factors:")
print(top_negative)

In [ ]:
for feature,value in top_positive.items():

    print(
        f"Higher {feature} increases default risk "
        f"(correlation {round(value,2)})"
    )